In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *

from lib.tables.CustomerTables import *
from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.ingester.ingester import *
from lib.query.bank import *

/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [18]:
#Get customer list
customer_query = 'SELECT * FROM KPI_Customer'
customer_list = Query(query = "SELECT * FROM KPI_Customer WHERE DBLocation IS NOT 'Unknown'").execute(KPIHub_Conn)

In [19]:
customer_list

,Id,Name,ShortName,DBLocation,LastUpdated
0,BD4D080B-1D12-D329-ABD0-39FEB9804E98,Cadent,Cadent,EU2,2026-05-20 12:54:11.021611
1,CFFB9000-94BD-BA72-B352-39EBA962116D,ITALGAS,ITALGAS,EU2,2026-05-20 12:54:11.086137
2,B511B372-18D7-E520-1328-39F05C8E531A,Toscana Energia,ToscanaEnergia,EU2,2026-05-20 12:54:11.152138
3,4A8FDDAD-482F-DF96-E655-3A065AC4817B,DEPA,DEPA,EU2,2026-05-20 12:54:11.221725


In [23]:
customer_name = customer_list.iloc[0]['Name']
customer_id = customer_list.iloc[0]['Id']
customer_db = customer_list.iloc[0]['DBLocation']
current_year = 2024
year_list = [year for year in range(STARTING_YEAR, current_year + 1)]
conn_dict = {'EU1':EU1_Conn, 'EU2': EU2_Conn}
a = get_reports(customer_name,years = year_list).execute(conn_dict[customer_db])

In [ ]:
query = Query(f"SELECT ReportDate FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}' ORDER BY ReportDate DESC LIMIT 1").execute(KPIHub_Conn)
if ~query.empty:
    last_report_year = query.iloc[0]['ReportDate']
else:
    last_report_year = STARTING_YEAR




IndexError: single positional indexer is out-of-bounds

In [5]:
a.db.set_query(query_reports_view(report_table = 'temp_reports'))
a.db.execute(DATAHUB_Conn, source_col = 'ReportId', temp_table_name = 'temp_reports')


,ReportId,ReportLabelOther,BoundaryName,BoundaryMode,BoundaryType,BoundaryPlant,BoundarySubplant,BoundaryRegion,BoundarySubRegion,BoundaryKmNetwork
0,A040C836-24AD-6BAF-8758-3A200F17E45C,emissions360,B24 - Wembley - Brent - Wembley Park,customer boundary,Wards 2024,Wembley,None,North London,Brent,12.470171
1,8C47241A-20D6-DBC5-8F99-3A1F20371EDC,emissions360,B25R2 - South Yorkshire - Sheffield - Broomhil...,customer boundary,Wards 2025 R2,South Yorkshire,None,East Midlands,Sheffield,51.814865
2,10179DEE-DF42-4315-466E-3A1F5E184342,emissions360,B25R2 - South Yorkshire - Sheffield - Nether E...,customer boundary,Wards 2025 R2,South Yorkshire,None,East Midlands,Sheffield,56.133052
3,2C3A35E4-246D-5899-A7D1-3A207E29461C,emissions360,B25R2 - Suffolk and Essex - West Suffolk - Bra...,customer boundary,Wards 2025 R2,Suffolk & Essex,None,East of England,West Suffolk,17.749447
4,8EB57F16-B204-C6D2-BB35-3A20407479C5,emissions360,B25R2 - Rayleigh - Havering - Hacton,customer boundary,Wards 2025 R2,Rayleigh,None,North London,Havering,30.767915
...,...,...,...,...,...,...,...,...,...,...
1228,712DF9F4-1068-39E9-5B24-3A1F533DBA50,emissions360,B25R2 - Warrington - Halton - Appleton,customer boundary,Wards 2025 R2,Warrington,None,North West,Halton,27.183186
1229,BEB8948C-4798-C85C-8C19-3A201C82D76D,emissions360,B25R2 - Black Country - Dudley - Sedgley,customer boundary,Wards 2025 R2,Black Country,None,West Midlands,Dudley,52.971442
1230,ABAC6BF1-258B-251D-4B7A-3A201235F56E,emissions360,B25R2 - Herts and Beds - North Hertfordshire -...,customer boundary,Wards 2025 R2,Herts & Beds,None,East of England,North Hertfordshire,12.166449
1231,847A0A7D-858A-1F2B-A8D0-3A20C11ACB39,emissions360,B26 - 6808 - South Yorkshire - Barnsley - Dear...,customer boundary,Wards 2026,South Yorkshire,None,East Midlands,Barnsley,46.441609
